In [1]:
import pandas as pd
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import our custom modules
from src.demand_model import AcceptanceProbabilityModel
from src.simulation import MonteCarloSimulation
from config.scenarios import SCENARIOS

In [2]:
# Load data and pre-trained model
processed_data_path = os.path.join(project_root, 'data', 'processed', 'loan_data_processed.csv')
df = pd.read_csv(processed_data_path)

demand_model_path = os.path.join(project_root, 'outputs', 'models', 'cox_ph_model.pkl')
loaded_demand_model = AcceptanceProbabilityModel.load_model(demand_model_path)

In [ ]:
# Configure and run the simulation
baseline_scenario = SCENARIOS['baseline']
n_sim_iterations = 100

mc_sim = MonteCarloSimulation(
    initial_df=df,
    demand_model=loaded_demand_model,
    scenario=baseline_scenario,
    n_iterations=n_sim_iterations
)

simulation_results = mc_sim.run_simulation(verbose=True)

Running Monte Carlo Simulation:  58%|█████▊    | 58/100 [00:43<00:31,  1.31it/s]

In [ ]:
# Analyze and display results
var_95 = simulation_results['total_npv'].quantile(0.05)
cvar_95 = simulation_results[simulation_results['total_npv'] <= var_95]['total_npv'].mean()
mean_npv = simulation_results['total_npv'].mean()
mean_defaults = simulation_results['total_defaults'].mean()

print("--- Key Performance Indicators ---")
print(f"  Expected Annual NPV: ${mean_npv:,.2f}")
print(f"  Value at Risk (VaR @ 95%): ${var_95:,.2f}")
print(f"  Conditional VaR (CVaR @ 95%): ${cvar_95:,.2f}")
print(f"  Average Annual Defaults: {mean_defaults:.2f}")

--- Key Performance Indicators ---
  Expected Annual NPV: $97,622.69
  Value at Risk (VaR @ 95%): $79,462.01
  Conditional VaR (CVaR @ 95%): $78,046.60
  Average Annual Defaults: 377.99


In [ ]:
# Run comparative scenario analysis
scenarios_to_compare = ['conservative', 'aggressive', 'no_optimization']
scenario_results = {'baseline': simulation_results}

for scenario_name in scenarios_to_compare:
    print(f"\n{'='*60}")
    print(f"Running {scenario_name.upper()} scenario...")
    print(f"{'='*60}")
    
    mc_sim_scenario = MonteCarloSimulation(
        initial_df=df,
        demand_model=loaded_demand_model,
        scenario=SCENARIOS[scenario_name],
        n_iterations=n_sim_iterations
    )
    
    results = mc_sim_scenario.run_simulation(verbose=True)
    scenario_results[scenario_name] = results
    
    # Quick summary
    mean_npv = results['total_npv'].mean()
    mean_defaults = results['total_defaults'].mean()
    mean_volume = results['total_increase_volume'].mean()
    print(f"  Expected NPV: ${mean_npv:,.2f}")
    print(f"  Avg Defaults: {mean_defaults:.2f}")
    print(f"  Avg Volume: ${mean_volume:,.2f}")

In [ ]:
# Visualize the distribution of Total NPV
plt.figure(figsize=(12, 7))
sns.histplot(simulation_results['total_npv'], bins=50, kde=True)
plt.axvline(mean_npv, color='red', linestyle='--', label=f'Mean NPV: ${mean_npv:,.2f}')
plt.axvline(var_95, color='purple', linestyle='--', label=f'VaR @ 95%: ${var_95:,.2f}')
plt.axvline(cvar_95, color='orange', linestyle='--', label=f'CVaR @ 95%: ${cvar_95:,.2f}')

plt.title(f'Distribution of Simulated Annual NPV (N={n_sim_iterations})')
plt.xlabel('Total Net Present Value ($)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Create comprehensive comparison visualizations

# 1. Comparative NPV Distribution (overlaid histograms)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-left: Overlaid distributions
ax1 = axes[0, 0]
for scenario_name, results in scenario_results.items():
    ax1.hist(results['total_npv'], bins=30, alpha=0.5, label=scenario_name)
ax1.set_xlabel('Total NPV ($)')
ax1.set_ylabel('Frequency')
ax1.set_title('NPV Distribution Comparison')
ax1.legend()
ax1.grid(alpha=0.3)

# Top-right: Box plot comparison
ax2 = axes[0, 1]
data_for_box = [results['total_npv'] for results in scenario_results.values()]
ax2.boxplot(data_for_box, labels=scenario_results.keys())
ax2.set_ylabel('Total NPV ($)')
ax2.set_title('NPV Distribution - Box Plot')
ax2.grid(alpha=0.3)

# Bottom-left: Defaults comparison
ax3 = axes[1, 0]
scenario_names = list(scenario_results.keys())
mean_defaults = [results['total_defaults'].mean() for results in scenario_results.values()]
ax3.bar(scenario_names, mean_defaults, color=['blue', 'green', 'red', 'orange'])
ax3.set_ylabel('Average Annual Defaults')
ax3.set_title('Default Volume Comparison')
ax3.grid(alpha=0.3)

# Bottom-right: Risk-Return scatter
ax4 = axes[1, 1]
mean_npvs = [results['total_npv'].mean() for results in scenario_results.values()]
std_npvs = [results['total_npv'].std() for results in scenario_results.values()]
ax4.scatter(std_npvs, mean_npvs, s=100)
for i, name in enumerate(scenario_names):
    ax4.annotate(name, (std_npvs[i], mean_npvs[i]), xytext=(5,5), textcoords='offset points')
ax4.set_xlabel('NPV Standard Deviation (Risk)')
ax4.set_ylabel('Mean NPV (Return)')
ax4.set_title('Risk-Return Frontier')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 2. Summary comparison table
comparison_df = pd.DataFrame({
    'Scenario': scenario_names,
    'Mean NPV': [f"${results['total_npv'].mean():,.2f}" for results in scenario_results.values()],
    'VaR (95%)': [f"${results['total_npv'].quantile(0.05):,.2f}" for results in scenario_results.values()],
    'CVaR (95%)': [f"${results[results['total_npv'] <= results['total_npv'].quantile(0.05)]['total_npv'].mean():,.2f}" for results in scenario_results.values()],
    'Avg Defaults': [f"{results['total_defaults'].mean():.2f}" for results in scenario_results.values()],
    'Avg Volume': [f"${results['total_increase_volume'].mean():,.0f}" for results in scenario_results.values()]
})

print("\n" + "="*80)
print("SCENARIO COMPARISON SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))